In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input, decode_predictions
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Add
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

data_dir = r"D:\ImageNet"
img_size = (224, 224)
batch_size = 32

ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, image_size=img_size, batch_size=batch_size,
    shuffle=True, seed=42
)

class_names = ds.class_names
captions = ["<start> a photo of a " + name.replace("_", " ").lower() + " <end>"
            for name in class_names]

tokenizer = Tokenizer(filters="", lower=True, oov_token="<unk>")
tokenizer.fit_on_texts(captions)
vocab_size = len(tokenizer.word_index) + 1
max_len = max(len(c.split()) for c in captions)

sequences, image_features = [], []
encoder = ResNet50(weights="imagenet", include_top=False, pooling="avg")

for images, labels in ds:
    x = preprocess_input(images.numpy())
    image_features.extend(encoder.predict(x, verbose=0))
    for label in labels.numpy():
        sequences.append(tokenizer.texts_to_sequences([captions[label]])[0])

X_img, X_seq, y = [], [], []
for i, seq in enumerate(sequences):
    for j in range(1, len(seq)):
        X_img.append(image_features[i])
        X_seq.append(pad_sequences([seq[:j]], maxlen=max_len-1)[0])
        y.append(seq[j])

X_img, X_seq, y = np.array(X_img), np.array(X_seq), np.array(y)

img_input = Input(shape=(2048,))
seq_input = Input(shape=(max_len-1,))
x = Embedding(vocab_size, 128, mask_zero=True)(seq_input)
x = LSTM(256)(x)
x = Add()([Dense(256)(img_input), x])
output = Dense(vocab_size, activation="softmax")(x)

model = Model([img_input, seq_input], output)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
model.fit([X_img, X_seq], y, epochs=10, batch_size=32)

def generate_caption(image, class_id):
    feature = encoder.predict(
        preprocess_input(np.expand_dims(image, 0)), verbose=0
    )
    seq = [tokenizer.word_index["<start>"]]
    for _ in range(max_len):
        padded = pad_sequences([seq], maxlen=max_len-1)
        pred = model.predict([feature, padded], verbose=0)
        word_id = np.argmax(pred[0])
        word = tokenizer.index_word.get(word_id, "")
        if word == "<end>" or not word:
            break
        seq.append(word_id)
    return " ".join(tokenizer.index_word.get(i, "") for i in seq[1:])

for images, labels in ds.take(1):
    for i in range(min(5, len(images))):
        image = images[i].numpy().astype("uint8")
        caption = generate_caption(image, labels[i].numpy())
        plt.imshow(image)
        plt.title("Expected: " + captions[labels[i].numpy()].replace("<start> ", "").replace(" <end>", "") +
                  "\nGenerated: " + caption)
        plt.axis("off")
        plt.show()